In [ ]:
#!/usr/bin/env python3
from __future__ import annotations

import os
import re
import math
from pathlib import Path
from typing import Any, Iterable, Mapping, Sequence

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

try:
    import scikit_posthocs as sp
except ImportError:
    sp = None


# ============================================================
# User configuration
# ============================================================

root_dir = Path(os.getcwd())

# Choose datasets/groups here.
dataset_names = [
    "Replogle_K562_essential",
    "Replogle_RPE",
    "NormanWeissman2019",
    "ChangYe",
    "ZhaoSims2021",
]

groups = ["single", "dual", "multi"]

# Treat each <dataset, group> pair as an independent benchmark dataset
# for dataset-level and summary CD plots. This means, for example,
# NormanWeissman2019/single and NormanWeissman2019/dual are ranked
# as two separate benchmark datasets.
TREAT_GROUP_AS_SEPARATE_DATASET = True
DATASET_GROUP_SEPARATOR = "__"

# If True, scan all valid <dataset>_pseudo_pairing_evaluation/<group> paths
# from dataset_names/groups. If False, only process SELECTED_SUB_PATH.
RUN_ALL_VALID_PATHS = True
SELECTED_SUB_PATH: Path | None = None

SELECTION_TABLE_NAME = "selected_variants_TEMPLATE_EDIT_ME.csv"

# Use exactly these pseudo-control variants in every CD analysis. The seed suffix
# identifies the selected pseudo-control dataset; repeated evaluation/model seeds
# are still retained as benchmark blocks for the significance tests.
USE_FIXED_VARIANTS = True
REQUIRE_ALL_FIXED_VARIANTS = True
VARIANTS = [
    "S0_naive_mean_control_reference",
    "S1_random_single_control/seed_000",
    "S2_random_average_controls/k_100/seed_000",
    "S3_SEACell_metacell_average/nmc_350/k_05/seed_000",
    "S4_SEACell_balanced_random_sample/nmc_350/seed_000",
    "S5_SEACell_OT_sampled_average/nmc_350/topk_05/seed_000",
]

# Labels used in the CD diagrams. The two S5 settings remain separate methods.
VARIANT_PLOT_LABELS = {
    "S0_naive_mean_control_reference": "Naive mean control",
    "S1_random_single_control/seed_000": "Random single control",
    "S2_random_average_controls/k_100/seed_000": "Random average control",
    "S3_SEACell_metacell_average/nmc_350/k_05/seed_000": "Random metacell average",
    "S4_SEACell_balanced_random_sample/nmc_350/seed_000": "SEACell balanced sample",
    "S5_SEACell_OT_sampled_average/nmc_350/topk_05/seed_000": "OT sampled average",
}

# Keep each selected variant as an independent method.
METHOD_LEVEL = "variant"  # "variant" or "strategy"

# S0 naive mean control is included as a competing method in CD diagrams
# so its rank directly shows how much selected pseudo-control strategies improve over it.
INCLUDE_S0_AS_METHOD = True
ALWAYS_INCLUDE_NAIVE_MEAN_CONTROL = True
NAIVE_BASELINE_ID = "S0_naive_mean_control_reference"

# Optional: keep S1 even if it is not selected, useful as a reference competitor.
ALWAYS_INCLUDE_RANDOM_SINGLE = False
RANDOM_SINGLE_ID = "S1_random_single_control"

# Strategy-level cleanup before CD plotting.
# - S4 is treated as one S4 method across all S4 parameter variants.
# - If multiple S5 variants were selected, keep only the best S5 variant per dataset/group.
# - After selecting the best S5 variant, S5 is collapsed to one strategy-level method.
# - Only S0 is excluded from the actual CD diagrams by default.
COLLAPSE_S4_VARIANTS_TO_BASE_METHOD = False
SELECT_BEST_S5_VARIANT = False
S5_BEST_SELECTION_SCOPE = "dataset_group"  # retained for compatibility; unused when False
COLLAPSE_S5_AFTER_BEST_SELECTION = False
USE_STRATEGY_PLOT_LABELS_FOR_CD = False
EXCLUDE_STRATEGIES_FROM_CD: list[str] = []

# Choose which metrics enter the critical-difference ranking.
# You can freely edit this list. Each name must exist in METRIC_DEFINITIONS below.
INCLUDED_METRICS = [
    # Control manifold preservation
    "control_local_mixing",
    "control_mmd_pca",

    # All-gene perturbation effect
    "perturbation_effect_rmse",
    "perturbation_effect_pearson",

    # Gene-program perturbation effect
    # "gene_program_rmse",
    # "gene_program_mae",
    # "gene_program_pearson",  
    # "gene_program_per_program_rmse_mean",

    # Forward MLP results
    "mlp_forward_model_mse",
    "mlp_forward_model_mae",
    "mlp_forward_model_common_reference_correlation",

    # Inverse MLP perturbation-identity results
    "mlp_inverse_model_test_accuracy",
    "mlp_inverse_model_macro_f1",
    "mlp_inverse_model_macro_precision",
    "mlp_inverse_model_macro_recall",
    "mlp_inverse_model_macro_auc",
]

# Ranking / post-hoc behavior. The significance workflow is unchanged:
# Friedman omnibus test followed by Conover-Friedman pairwise tests with Holm adjustment.
# Ranks are computed within each benchmark block. Lower rank = better.
# The scikit-posthocs critical_difference_diagram receives:
#   ranks      = average rank per selected strategy/variant
#   sig_matrix = Conover-Friedman pairwise p-value matrix computed from rank blocks
ALPHA = 0.05

# scikit-posthocs CD diagrams require a complete block-by-method matrix.
# If different datasets select different variants, the script will greedily keep
# the largest method subset that has enough complete ranking blocks.
COMPLETE_BLOCKS_ONLY = True
MIN_BLOCKS_FOR_CD = 2
MIN_METHODS_FOR_CD = 2
MIN_METHOD_COVERAGE_FRACTION = 0.0
# Do not drop these methods when choosing a complete design, if present.
# Instead, blocks where these methods are missing are removed.
FORCE_KEEP_METHOD_IDS = [NAIVE_BASELINE_ID]

# Strict fixed-variant mode: never remove one of the seven requested variants
# to manufacture a complete design. Instead, keep only benchmark blocks that
# contain all seven methods; if fewer than MIN_BLOCKS_FOR_CD remain, stop with
# a diagnostic error rather than drawing a six-method CD diagram.
FORCE_KEEP_ALL_FIXED_VARIANTS = True

# Which CD plots to create.
# 1) For each dataset/group and each metric: blocks = random seeds.
MAKE_DATASET_METRIC_CD = True

# 2) For each dataset/group: blocks = metrics after averaging random seeds.
MAKE_DATASET_SUMMARY_CD = True

# 3) Across all dataset/groups: blocks = dataset/group × metric after averaging random seeds.
MAKE_ALL_DATASETS_SUMMARY_CD = True

# Optional extra: across all dataset/groups directly using seed-level blocks.
MAKE_ALL_DATASETS_SEED_LEVEL_CD = False

# If True, the script also saves the long metric table, mean table,
# average ranks, rank matrices, and Conover p-value matrices.
SAVE_INTERMEDIATE_TABLES = True

# Plot output.
OUTPUT_DIR = root_dir / "critical_difference_final"
SAVE_PNG = True
SAVE_SVG = True
DPI = 300
SHOW_FIGURES = False

# Plot appearance for scikit_posthocs.critical_difference_diagram.
CD_FIG_WIDTH_BASE = 10.0
CD_FIG_HEIGHT_BASE = 3.2
CD_FIG_HEIGHT_PER_METHOD = 0.24
CD_TITLE_SIZE = 13.0
CD_SUBTITLE_SIZE = 9.5
CD_LABEL_SIZE = 9.0
CD_MARKER_SIZE = 42
CD_LINEWIDTH = 1.1
CD_CROSSBAR_LINEWIDTH = 3.0
CD_COLOR = "#222222"
CD_CROSSBAR_COLOR = "#333333"


# ============================================================
# Metric definitions
# ============================================================

# source:
#   - "control_manifold": seed-level control metric table
#   - "perturbation_effect": all-gene perturbation-effect seed table
#   - "gene_program": gene-program seed-level metric table
#   - "mlp_forward": forward MLP repeated-run summary table
#   - "mlp_inverse": inverse MLP repeated-run summary table
#
# direction:
#   - "lower": smaller values rank better
#   - "higher": larger values rank better
#
# selected_mean_columns are used only if no seed-level source table is available.
METRIC_DEFINITIONS: dict[str, dict[str, Any]] = {
    "control_local_mixing": {
        "label": "Local mixing score",
        "source": "control_manifold",
        "direction": "higher",
        "source_columns": [
            "control_pseudo_local_mixing_score",
            "source_mixing_opposite_neighbor_fraction_mean",
            "local_mixing_score",
        ],
        "selected_mean_columns": ["control_manifold__control_pseudo_local_mixing_score_mean"],
        "selected_std_columns": ["control_manifold__control_pseudo_local_mixing_score_std"],
        "selected_n_columns": ["control_manifold__control_pseudo_local_mixing_score_n"],
    },
    "control_mmd_pca": {
        "label": "MMD in PCA space",
        "source": "control_manifold",
        "direction": "lower",
        "source_columns": ["mmd_pca", "mmd_rbf_pca", "mmd_pca_mean"],
        "selected_mean_columns": ["control_manifold__mmd_pca_mean"],
        "selected_std_columns": ["control_manifold__mmd_pca_std"],
        "selected_n_columns": ["control_manifold__mmd_pca_n"],
    },
    "perturbation_effect_rmse": {
        "label": "All-gene perturbation-effect RMSE",
        "source": "perturbation_effect",
        "direction": "lower",
        "source_columns": [
            "strategy_delta_rmse_common_mean",
            "perturbation_effect_rmse",
            "perturbation_effect__perturbation_effect_rmse_mean",
        ],
        "selected_mean_columns": ["perturbation_effect__perturbation_effect_rmse_mean"],
        "selected_std_columns": ["perturbation_effect__perturbation_effect_rmse_std"],
        "selected_n_columns": ["perturbation_effect__perturbation_effect_rmse_n"],
    },
    "perturbation_effect_pearson": {
        "label": "All-gene perturbation-effect Pearson",
        "source": "perturbation_effect",
        "direction": "higher",
        "source_columns": [
            "strategy_delta_pearson_common_mean",
            "perturbation_effect_pearson",
            "perturbation_effect__perturbation_effect_pearson_mean",
        ],
        "selected_mean_columns": ["perturbation_effect__perturbation_effect_pearson_mean"],
        "selected_std_columns": ["perturbation_effect__perturbation_effect_pearson_std"],
        "selected_n_columns": ["perturbation_effect__perturbation_effect_pearson_n"],
    },
    # "gene_program_rmse": {
    #     "label": "Gene-program effect RMSE",
    #     "source": "gene_program",
    #     "direction": "lower",
    #     "source_columns": ["rmse", "gene_program_rmse"],
    # },
    # "gene_program_mae": {
    #     "label": "Gene-program effect MAE",
    #     "source": "gene_program",
    #     "direction": "lower",
    #     "source_columns": ["mae", "gene_program_mae"],
    # },
    # "gene_program_pearson": {
    #     "label": "Gene-program effect Pearson",
    #     "source": "gene_program",
    #     "direction": "higher",
    #     "source_columns": ["pearson", "gene_program_pearson"],
    # },
    # "gene_program_per_program_rmse_mean": {
    #     "label": "Per-program RMSE mean",
    #     "source": "gene_program",
    #     "direction": "lower",
    #     "source_columns": ["per_program_rmse_mean"],
    # },


    "mlp_forward_model_mse": {
        "label": "Forward MLP model MSE",
        "source": "mlp_forward",
        "direction": "lower",
        "source_columns": [
            "model_mse_xt",
            "model_mse",
            "test_mse",
            "mse",
            "mlp_forward__model_mse_mean",
        ],
        "selected_mean_columns": ["mlp_forward__model_mse_mean"],
        "selected_std_columns": ["mlp_forward__model_mse_std"],
        "selected_n_columns": ["mlp_forward__model_mse_n"],
    },
    "mlp_forward_model_mae": {
        "label": "Forward MLP model MAE",
        "source": "mlp_forward",
        "direction": "lower",
        "source_columns": [
            "model_mae_xt",
            "model_mae",
            "test_mae",
            "mae",
            "mlp_forward__model_mae_mean",
        ],
        "selected_mean_columns": ["mlp_forward__model_mae_mean"],
        "selected_std_columns": ["mlp_forward__model_mae_std"],
        "selected_n_columns": ["mlp_forward__model_mae_n"],
    },
    # "mlp_forward_model_common_reference_correlation": {
    #     "label": "Forward MLP model common-reference correlation",
    #     "source": "mlp_forward",
    #     "direction": "higher",
    #     "source_columns": [
    #         # Seed-level forward MLP output.
    #         "model_common_delta_cell_pearson_mean",
    #         # Aggregated result-analysis aliases.
    #         "model_common_reference_correlation",
    #         "model_common_reference_correlation_mean",
    #         "model_common_delta_pearson_mean",
    #         # Selected-variant table fallback.
    #         "mlp_forward__model_common_reference_correlation_mean",
    #     ],
    #     "selected_mean_columns": [
    #         "mlp_forward__model_common_reference_correlation_mean",
    #     ],
    #     "selected_std_columns": [
    #         "mlp_forward__model_common_reference_correlation_std",
    #     ],
    #     "selected_n_columns": [
    #         "mlp_forward__model_common_reference_correlation_n",
    #     ],
    # },
    "mlp_inverse_model_test_accuracy": {
        "label": "Inverse MLP test accuracy",
        "source": "mlp_inverse",
        "direction": "higher",
        "source_columns": [
            "test_accuracy",
            "accuracy",
            "model_accuracy",
            "inverse_accuracy",
            "strategy_delta_test_accuracy",
            "strategy_delta_accuracy",
            "mlp_inverse__test_accuracy_mean",
        ],
        "selected_mean_columns": ["mlp_inverse__test_accuracy_mean"],
        "selected_std_columns": ["mlp_inverse__test_accuracy_std"],
        "selected_n_columns": ["mlp_inverse__test_accuracy_n"],
    },
    # "mlp_inverse_model_macro_f1": {
    #     "label": "Inverse MLP macro F1",
    #     "source": "mlp_inverse",
    #     "direction": "higher",
    #     "source_columns": [
    #         "test_macro_f1",
    #         "macro_f1",
    #         "model_macro_f1",
    #         "inverse_macro_f1",
    #         "strategy_delta_test_macro_f1",
    #         "strategy_delta_macro_f1",
    #         "mlp_inverse__macro_f1_mean",
    #     ],
    #     "selected_mean_columns": ["mlp_inverse__macro_f1_mean"],
    #     "selected_std_columns": ["mlp_inverse__macro_f1_std"],
    #     "selected_n_columns": ["mlp_inverse__macro_f1_n"],
    # },
    # "mlp_inverse_model_macro_precision": {
    #     "label": "Inverse MLP macro precision",
    #     "source": "mlp_inverse",
    #     "direction": "higher",
    #     "source_columns": [
    #         "test_macro_precision",
    #         "macro_precision",
    #         "model_macro_precision",
    #         "inverse_macro_precision",
    #         "strategy_delta_test_macro_precision",
    #         "strategy_delta_macro_precision",
    #         "precision",
    #         "mlp_inverse__precision_mean",
    #     ],
    #     "selected_mean_columns": ["mlp_inverse__precision_mean"],
    #     "selected_std_columns": ["mlp_inverse__precision_std"],
    #     "selected_n_columns": ["mlp_inverse__precision_n"],
    # },
    # "mlp_inverse_model_macro_recall": {
    #     "label": "Inverse MLP macro recall",
    #     "source": "mlp_inverse",
    #     "direction": "higher",
    #     "source_columns": [
    #         "test_macro_recall",
    #         "macro_recall",
    #         "model_macro_recall",
    #         "inverse_macro_recall",
    #         "strategy_delta_test_macro_recall",
    #         "strategy_delta_macro_recall",
    #         "recall",
    #         "mlp_inverse__recall_mean",
    #     ],
    #     "selected_mean_columns": ["mlp_inverse__recall_mean"],
    #     "selected_std_columns": ["mlp_inverse__recall_std"],
    #     "selected_n_columns": ["mlp_inverse__recall_n"],
    # },
    "mlp_inverse_model_macro_auc": {
        "label": "Inverse MLP macro AUC (OvR)",
        "source": "mlp_inverse",
        "direction": "higher",
        "source_columns": [
            # Seed-level inverse MLP output.
            "test_macro_auc_ovr",
            # Common aliases used by aggregated tables.
            "test_macro_auc",
            "macro_auc_ovr",
            "macro_auc",
            "model_macro_auc",
            "inverse_macro_auc",
            "strategy_delta_test_macro_auc_ovr",
            "strategy_delta_macro_auc",
            # Selected-variant table fallback.
            "mlp_inverse__macro_auc_mean",
        ],
        "selected_mean_columns": ["mlp_inverse__macro_auc_mean"],
        "selected_std_columns": ["mlp_inverse__macro_auc_std"],
        "selected_n_columns": ["mlp_inverse__macro_auc_n"],
    },
    # "mlp_inverse_accuracy": {
    #     "label": "Inverse MLP accuracy",
    #     "source": "mlp_inverse",
    #     "direction": "higher",
    #     "source_columns": ["test_accuracy", "accuracy", "mlp_inverse__test_accuracy_mean"],
    #     "selected_mean_columns": ["mlp_inverse__test_accuracy_mean"],
    #     "selected_std_columns": ["mlp_inverse__test_accuracy_std"],
    #     "selected_n_columns": ["mlp_inverse__test_accuracy_n"],
    # },
    # "mlp_inverse_macro_f1": {
    #     "label": "Inverse MLP macro F1",
    #     "source": "mlp_inverse",
    #     "direction": "higher",
    #     "source_columns": ["test_macro_f1", "macro_f1", "mlp_inverse__macro_f1_mean"],
    #     "selected_mean_columns": ["mlp_inverse__macro_f1_mean"],
    #     "selected_std_columns": ["mlp_inverse__macro_f1_std"],
    #     "selected_n_columns": ["mlp_inverse__macro_f1_n"],
    # },
}


# ============================================================
# Source table locations
# ============================================================

CONTROL_MANIFOLD_SEED_TABLE_CANDIDATES = [
    Path("result_analysis") / "aggregated_by_task" / "control_manifold" / "control_manifold_canonical_input_with_required_metrics.csv",
    Path("control_manifold") / "control_manifold_preservation_repeated_run_summary.csv",
    Path("control_manifold") / "control_manifold_preservation_repeated_summary.csv",
    Path("control_manifold") / "control_manifold_preservation_repeated_long.csv",
]

PERTURBATION_EFFECT_SEED_TABLE_CANDIDATES = [
    Path("perturbation_effect") / "perturbation_effect_consistency_repeated_run_summary.csv",
    Path("perturbation_effect") / "perturbation_effect_consistency_repeated_run_summary_PARTIAL.csv",
    Path("result_analysis") / "aggregated_by_task" / "perturbation_effect" / "perturbation_effect_consistency_repeated_run_summary.csv",
]

GENE_PROGRAM_SEED_TABLE_CANDIDATES = [
    Path("result_analysis") / "gene_program_level" / "metrics" / "gene_program_metrics_by_variant_seed_level.csv",
    Path("result_analysis") / "gene_program_level" / "metrics" / "gene_program_metrics_by_variant.csv",
]

MLP_FORWARD_SEED_TABLE_CANDIDATES = [
    Path("downstream_mlp") / "forward_mlp_run_summary.csv",
    Path("downstream_mlp") / "forward_mlp_repeated_run_summary.csv",
    Path("result_analysis") / "aggregated_by_task" / "mlp_forward" / "forward_mlp_run_summary.csv",
    Path("result_analysis") / "aggregated_by_task" / "mlp_forward" / "forward_mlp_repeated_run_summary.csv",
]

MLP_INVERSE_SEED_TABLE_CANDIDATES = [
    Path("downstream_mlp") / "inverse_mlp_run_summary.csv",
    Path("downstream_mlp") / "inverse_mlp_repeated_strategy_delta_classification_summary.csv",
    Path("downstream_mlp") / "inverse_mlp_repeated_run_summary.csv",
    Path("result_analysis") / "aggregated_by_task" / "mlp_inverse" / "inverse_mlp_run_summary.csv",
    Path("result_analysis") / "aggregated_by_task" / "mlp_inverse" / "inverse_mlp_repeated_strategy_delta_classification_summary.csv",
    Path("result_analysis") / "aggregated_by_task" / "mlp_inverse" / "inverse_mlp_repeated_run_summary.csv",
]


# ============================================================
# Strategy labels and variant canonicalization
# ============================================================

STRATEGY_PLOT_LABELS = {
    "S0_naive_mean_control_reference": "Naive mean control",
    "S1_random_single_control": "Random single control",
    "S2_random_average_controls": "Random average control",
    "S3_SEACell_metacell_average": "Random metacell average",
    "S4_SEACell_balanced_random_sample": "SEACell balanced sample",
    "S5_SEACell_OT_sampled_average": "OT sampled average",
}

DEFAULT_STRATEGY_RENAME_MAP = {
    "S0_naive_mean_control_reference": "S0_naive_mean_control_reference",
    "S1_random_single_control": "S1_random_single_control",
    "S2_random_average_controls": "S2_random_average_controls",
    "S3_SEACell_metacell_average": "S3_SEACell_metacell_average",
    "S4_SEACell_balanced_random_sample": "S4_SEACell_balanced_random_sample",
    "S5_SEACell_OT_sampled_average": "S5_SEACell_OT_sampled_average",
    "S0": "S0_naive_mean_control_reference",
    "S1": "S1_random_single_control",
    "S2": "S2_random_average_controls",
    "S3": "S3_SEACell_metacell_average",
    "S4": "S4_SEACell_balanced_random_sample",
    "S5": "S5_SEACell_OT_sampled_average",
    "S4_random_single_control_oracle": "S1_random_single_control",
    "S4_random_single_control": "S1_random_single_control",
    "strategy4_random_single_control": "S1_random_single_control",
    "strategy4_random_single_control_cell": "S1_random_single_control",
    "S3_random_average_controls": "S2_random_average_controls",
    "strategy3_random_average_controls": "S2_random_average_controls",
    "strategy3_random_average_control_cells": "S2_random_average_controls",
    "S5_random_metacell_average": "S3_SEACell_metacell_average",
    "S3_random_metacell_average": "S3_SEACell_metacell_average",
    "strategy5_random_metacell_average": "S3_SEACell_metacell_average",
    "S1_SEACell_balanced_random": "S4_SEACell_balanced_random_sample",
    "S4_SEACell_balanced_random": "S4_SEACell_balanced_random_sample",
    "strategy1_seacell_balanced_random_repeated": "S4_SEACell_balanced_random_sample",
    "S2_SEACell_OT_topk_sampled_average": "S5_SEACell_OT_sampled_average",
    "S2_SEACell_OT_topk_sampled_average_repeated": "S5_SEACell_OT_sampled_average",
    "strategy2_seacells_ot_topk_sampled_average": "S5_SEACell_OT_sampled_average",
    "strategy2_seacell_ot_topk_sampled_average_repeated": "S5_SEACell_OT_sampled_average",
}

STRATEGY_ORDER = {
    "S0_naive_mean_control_reference": 0,
    "S1_random_single_control": 1,
    "S2_random_average_controls": 2,
    "S3_SEACell_metacell_average": 3,
    "S4_SEACell_balanced_random_sample": 4,
    "S5_SEACell_OT_sampled_average": 5,
}


# ============================================================
# Generic helpers
# ============================================================

def is_missing(x: Any) -> bool:
    if x is None:
        return True
    try:
        return bool(pd.isna(x))
    except Exception:
        return False


def safe_filename(x: Any) -> str:
    s = str(x)
    s = re.sub(r"[^A-Za-z0-9_.+-]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s or "unnamed"


def read_table(path: str | Path) -> pd.DataFrame:
    """Read CSV/TSV/text tables robustly.

    Selection templates are comma-separated CSV files. During transfer or manual
    inspection they may occasionally have a .txt extension, so delimiter choice
    must not depend only on the suffix.
    """
    path = Path(path)
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path)

    if path.suffix.lower() in {".csv", ".tsv", ".txt"}:
        # Let pandas infer comma versus tab from the file contents.
        return pd.read_csv(path, sep=None, engine="python")

    return pd.read_csv(path, sep=None, engine="python")


def first_existing_path(paths: Iterable[Path]) -> Path | None:
    for p in paths:
        if p.exists():
            return p
    return None


def as_bool_series(s: pd.Series) -> pd.Series:
    if s.dtype == bool:
        return s.fillna(False)
    return s.astype(str).str.strip().str.lower().isin({"true", "1", "yes", "y", "t"})


def clean_label(x: Any) -> str:
    s = str(x)
    s = s.replace("_", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s


def to_float_or_nan(x: Any) -> float:
    try:
        val = float(x)
    except Exception:
        return np.nan
    if not np.isfinite(val):
        return np.nan
    return val


def format_number_for_key(x: Any) -> str | None:
    val = to_float_or_nan(x)
    if not np.isfinite(val):
        return None
    if abs(val - round(val)) < 1e-8:
        return str(int(round(val)))
    return f"{val:g}"


def infer_strategy_from_text(text: str) -> str | None:
    for key in STRATEGY_ORDER:
        if key in text:
            return key
    for short, full in DEFAULT_STRATEGY_RENAME_MAP.items():
        if short and short in text:
            return full
    return None


def extract_first_number(text: str, patterns: Sequence[str]) -> float:
    text = str(text)
    for pat in patterns:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            try:
                return float(m.group(1))
            except Exception:
                pass
    return np.nan


def coalesce_numeric_from_columns(row: pd.Series, columns: Sequence[str]) -> float:
    for col in columns:
        if col in row.index:
            val = to_float_or_nan(row[col])
            if np.isfinite(val):
                return val
    return np.nan


def canonicalize_variant_table(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    if "variant_id" not in out.columns:
        if "strategy_variant" in out.columns:
            out["variant_id"] = out["strategy_variant"]
        elif "strategy_id" in out.columns:
            out["variant_id"] = out["strategy_id"]
        elif "strategy" in out.columns:
            out["variant_id"] = out["strategy"]
        else:
            raise ValueError("Table must contain variant_id, strategy_variant, strategy_id, or strategy.")

    out["source_variant_id"] = out["variant_id"].astype(str)

    if "strategy" not in out.columns:
        out["strategy"] = out["source_variant_id"].map(lambda x: infer_strategy_from_text(str(x)) or str(x).split("__")[0])
    out["strategy"] = out["strategy"].astype(str).map(lambda x: DEFAULT_STRATEGY_RENAME_MAP.get(x, x))

    # Infer strategy again from variant_id if the strategy column is non-canonical.
    inferred = out["source_variant_id"].map(lambda x: infer_strategy_from_text(str(x)))
    out["strategy"] = [inf if inf is not None else st for st, inf in zip(out["strategy"], inferred)]

    if "strategy_order" not in out.columns:
        out["strategy_order"] = out["strategy"].map(STRATEGY_ORDER).fillna(999)
    else:
        out["strategy_order"] = pd.to_numeric(out["strategy_order"], errors="coerce")
        out["strategy_order"] = out["strategy_order"].where(out["strategy_order"].notna(), out["strategy"].map(STRATEGY_ORDER))
        out["strategy_order"] = out["strategy_order"].fillna(999)

    text_cols = []
    for col in ["source_variant_id", "variant_id", "display_variant_label", "variant_label", "parameter_label", "outdir", "pseudo_control_h5ad"]:
        if col in out.columns:
            text_cols.append(out[col].astype(str))
    text = text_cols[0] if text_cols else out["source_variant_id"].astype(str)
    for s in text_cols[1:]:
        text = text.str.cat(s, sep=" ")

    numeric_cols = [
        "n_metacells", "n_metacells_requested", "n_metacells_observed",
        "top_k", "top_k_metacells", "sampled_metacells_k", "n_metacells_to_average",
        "n_control_cells_to_average", "sample_cells_per_metacell",
    ]
    for col in numeric_cols:
        if col not in out.columns:
            out[col] = np.nan
        out[col] = pd.to_numeric(out[col], errors="coerce")

    # Fill common parameters from text when missing.
    missing = out["n_metacells"].isna()
    out.loc[missing, "n_metacells"] = pd.to_numeric(
        text.loc[missing].map(
            lambda x: extract_first_number(x, [r"nmc[_=-]?(\d+)", r"n[_=-]?metacells[_=-]?(\d+)"])
        ),
        errors="coerce",
    ).to_numpy()

    missing = out["top_k"].isna()
    out.loc[missing, "top_k"] = pd.to_numeric(
        text.loc[missing].map(
            lambda x: extract_first_number(x, [r"topk[_=-]?(\d+)", r"top[_=-]?k[_=-]?(\d+)"])
        ),
        errors="coerce",
    ).to_numpy()

    missing = out["sampled_metacells_k"].isna()
    out.loc[missing, "sampled_metacells_k"] = pd.to_numeric(
        text.loc[missing].map(
            lambda x: extract_first_number(x, [r"sampledMC[_=-]?(\d+)", r"sampled_metacells[_=-]?(\d+)"])
        ),
        errors="coerce",
    ).to_numpy()

    # S3 may encode the number of averaged/sampled metacells as either
    #   S3...__nmc_350__sampledMC_10
    #   S3...__nmc_350__k_10
    #   S3.../nmc_350/k_10/seed_000
    # The previous expression only recognized the double-underscore form, which
    # caused fixed slash-form variants to fail matching the template CSV.
    s3_missing = out["sampled_metacells_k"].isna() & out["strategy"].eq("S3_SEACell_metacell_average")
    out.loc[s3_missing, "sampled_metacells_k"] = pd.to_numeric(
        text.loc[s3_missing].map(
            lambda x: extract_first_number(
                x,
                [
                    r"sampledMC[_=-]?(\d+)",
                    r"sampled_metacells[_=-]?(\d+)",
                    r"(?:^|[/_])k[_=-]?(\d+)",
                ],
            )
        ),
        errors="coerce",
    ).to_numpy()

    # S2 may encode the number of averaged control cells as either __k_100
    # or /k_100/. Accept both representations.
    missing = out["n_control_cells_to_average"].isna()
    out.loc[missing, "n_control_cells_to_average"] = pd.to_numeric(
        text.loc[missing].map(
            lambda x: extract_first_number(
                x,
                [
                    r"cells[_=-]?(\d+)",
                    r"nctrl[_=-]?(\d+)",
                    r"(?:^|[/_])k[_=-]?(\d+)",
                ],
            )
        ),
        errors="coerce",
    ).to_numpy()

    missing = out["sample_cells_per_metacell"].isna()
    out.loc[missing, "sample_cells_per_metacell"] = pd.to_numeric(
        text.loc[missing].map(
            lambda x: extract_first_number(x, [r"spm[_=-]?(\d+)", r"sample_cells_per_metacell[_=-]?(\d+)"])
        ),
        errors="coerce",
    ).to_numpy()

    out["variant_match_key"] = out.apply(make_variant_match_key, axis=1)
    return out


def make_variant_match_key(row: pd.Series) -> str:
    strategy = str(row.get("strategy", row.get("source_variant_id", "")))
    strategy = DEFAULT_STRATEGY_RENAME_MAP.get(strategy, strategy)

    if strategy in {"S0_naive_mean_control_reference", "S1_random_single_control"}:
        return strategy

    if strategy == "S2_random_average_controls":
        k = format_number_for_key(row.get("n_control_cells_to_average"))
        return f"{strategy}__k_{k}" if k else strategy

    if strategy == "S3_SEACell_metacell_average":
        nmc = format_number_for_key(row.get("n_metacells"))
        k = format_number_for_key(row.get("sampled_metacells_k")) or format_number_for_key(row.get("n_metacells_to_average"))
        parts = [strategy]
        if nmc:
            parts.append(f"nmc_{nmc}")
        if k:
            parts.append(f"k_{k}")
        return "__".join(parts)

    if strategy == "S4_SEACell_balanced_random_sample":
        # For CD comparison, all S4 parameter variants are treated as one S4 method.
        # Returning the base strategy here also makes source-table/selected-table
        # matching robust when the exact S4 n_metacells label differs across tables.
        if COLLAPSE_S4_VARIANTS_TO_BASE_METHOD:
            return strategy
        nmc = format_number_for_key(row.get("n_metacells"))
        parts = [strategy]
        if nmc:
            parts.append(f"nmc_{nmc}")
        return "__".join(parts)

    if strategy == "S5_SEACell_OT_sampled_average":
        nmc = format_number_for_key(row.get("n_metacells"))
        topk = format_number_for_key(row.get("top_k")) or format_number_for_key(row.get("top_k_metacells"))
        parts = [strategy]
        if nmc:
            parts.append(f"nmc_{nmc}")
        if topk:
            parts.append(f"topk_{topk}")
        return "__".join(parts)

    return str(row.get("source_variant_id", row.get("variant_id", strategy)))


def extract_variant_suffix(row: pd.Series) -> str:
    strategy = str(row.get("strategy", ""))
    if strategy == "S2_random_average_controls":
        k = format_number_for_key(row.get("n_control_cells_to_average"))
        return f"({k} cells)" if k else ""
    if strategy == "S3_SEACell_metacell_average":
        nmc = format_number_for_key(row.get("n_metacells"))
        k = format_number_for_key(row.get("sampled_metacells_k")) or format_number_for_key(row.get("n_metacells_to_average"))
        return f"({nmc}&{k})" if nmc and k else ""
    if strategy == "S4_SEACell_balanced_random_sample":
        if COLLAPSE_S4_VARIANTS_TO_BASE_METHOD:
            return ""
        nmc = format_number_for_key(row.get("n_metacells"))
        return f"({nmc})" if nmc else ""
    if strategy == "S5_SEACell_OT_sampled_average":
        nmc = format_number_for_key(row.get("n_metacells"))
        topk = format_number_for_key(row.get("top_k")) or format_number_for_key(row.get("top_k_metacells"))
        return f"({nmc}&{topk})" if nmc and topk else ""
    return ""


def get_method_label(row: pd.Series) -> str:
    strategy = str(row.get("strategy", ""))
    match_key = str(row.get("variant_match_key", ""))

    # Exact labels for the fixed selected variants take priority. This preserves
    # separate labels for the two S5 variants while keeping concise strategy names.
    if USE_FIXED_VARIANTS and match_key in FIXED_LABEL_BY_MATCH_KEY:
        return FIXED_LABEL_BY_MATCH_KEY[match_key]

    # For CD diagrams, use the same clean strategy labels as the other figures.
    # This avoids variant-id labels such as __nmc_350__topk_5 in the CD diagram.
    if USE_STRATEGY_PLOT_LABELS_FOR_CD:
        return STRATEGY_PLOT_LABELS.get(strategy, strategy)

    if METHOD_LEVEL == "strategy":
        return STRATEGY_PLOT_LABELS.get(strategy, strategy)

    if COLLAPSE_S4_VARIANTS_TO_BASE_METHOD and strategy == "S4_SEACell_balanced_random_sample":
        return STRATEGY_PLOT_LABELS.get(strategy, strategy)

    for col in ["final_strategy_label", "plot_label"]:
        if col in row.index and not is_missing(row[col]) and str(row[col]).strip():
            return clean_label(row[col])

    base = STRATEGY_PLOT_LABELS.get(str(row.get("strategy", "")), str(row.get("strategy", "")))
    suffix = extract_variant_suffix(row)
    return clean_label(f"{base} {suffix}" if suffix else base)


def get_method_id(row: pd.Series) -> str:
    strategy = str(row.get("strategy", ""))
    if METHOD_LEVEL == "strategy":
        return str(row.get("strategy", row.get("variant_match_key", "method")))
    if COLLAPSE_S4_VARIANTS_TO_BASE_METHOD and strategy == "S4_SEACell_balanced_random_sample":
        return "S4_SEACell_balanced_random_sample"
    return str(row.get("variant_match_key", row.get("variant_id", "method")))


def infer_seed_column(df: pd.DataFrame) -> str:
    for col in ["sampling_seed", "seed", "pair_selection_seed", "random_seed", "run_seed", "training_seed", "model_seed", "split_seed", "sampling_seed_for_plot"]:
        if col in df.columns:
            return col
    df["_pseudo_seed_index"] = df.groupby("variant_match_key").cumcount()
    return "_pseudo_seed_index"


def filter_table_to_context(df: pd.DataFrame, dataset_id: str, group: str) -> pd.DataFrame:
    out = df.copy()
    if "dataset_id" in out.columns:
        out = out[out["dataset_id"].astype(str).eq(str(dataset_id))].copy()
    if "perturbed_group" in out.columns:
        out = out[out["perturbed_group"].astype(str).eq(str(group))].copy()
    return out


def make_dataset_group_id(dataset_id: str, group: str) -> str:
    if not TREAT_GROUP_AS_SEPARATE_DATASET:
        return str(dataset_id)
    return f"{dataset_id}{DATASET_GROUP_SEPARATOR}{group}"


def make_dataset_group_label(dataset_id: str, group: str) -> str:
    if not TREAT_GROUP_AS_SEPARATE_DATASET:
        return str(dataset_id)
    return f"{dataset_id} / {group}"


def normalize_variant_path(value: Any) -> str:
    """Normalize a slash/slug variant identifier for lookup and display."""
    s = str(value).strip().strip("/")
    s = s.replace("__", "/")
    s = re.sub(r"/+", "/", s)
    return s


def build_fixed_variant_table() -> pd.DataFrame:
    """Convert VARIANTS into the same canonical matching keys as source tables."""
    rows = []
    for order, variant_id in enumerate(VARIANTS):
        normalized = normalize_variant_path(variant_id)
        strategy = infer_strategy_from_text(normalized)
        if strategy is None:
            raise ValueError(f"Cannot infer strategy from fixed variant: {variant_id}")
        rows.append({
            "variant_id": normalized,
            "strategy": strategy,
            "fixed_variant_id": normalized,
            "fixed_variant_order": order,
            "fixed_method_label": VARIANT_PLOT_LABELS.get(normalized, normalized),
        })

    fixed = canonicalize_variant_table(pd.DataFrame(rows))
    fixed["method_id"] = fixed["variant_match_key"].astype(str)
    fixed["method_label"] = fixed["fixed_method_label"].astype(str)
    return fixed


FIXED_VARIANT_TABLE = build_fixed_variant_table()
FIXED_LABEL_BY_MATCH_KEY = (
    FIXED_VARIANT_TABLE
    .drop_duplicates("variant_match_key")
    .set_index("variant_match_key")["fixed_method_label"]
    .astype(str)
    .to_dict()
)
FIXED_ORDER_BY_MATCH_KEY = (
    FIXED_VARIANT_TABLE
    .drop_duplicates("variant_match_key")
    .set_index("variant_match_key")["fixed_variant_order"]
    .astype(int)
    .to_dict()
)


# ============================================================
# Loading selected variants and metrics
# ============================================================

def sub_path_for(dataset_id: str, group: str) -> Path:
    return root_dir / f"{dataset_id}_pseudo_pairing_evaluation" / group


def discover_sub_paths() -> list[tuple[str, str, Path]]:
    if not RUN_ALL_VALID_PATHS:
        if SELECTED_SUB_PATH is None:
            raise ValueError("SELECTED_SUB_PATH must be set when RUN_ALL_VALID_PATHS=False.")
        p = Path(SELECTED_SUB_PATH)
        group = p.name
        dataset_id = p.parent.name.replace("_pseudo_pairing_evaluation", "")
        return [(dataset_id, group, p)]

    out: list[tuple[str, str, Path]] = []
    for dataset_id in dataset_names:
        for group in groups:
            p = sub_path_for(dataset_id, group)
            selection_path = p / "result_analysis" / SELECTION_TABLE_NAME
            if selection_path.exists():
                out.append((dataset_id, group, p))
    return out


def load_selected_variants(selection_path: Path) -> pd.DataFrame:
    full = canonicalize_variant_table(read_table(selection_path))

    if USE_FIXED_VARIANTS:
        fixed_cols = [
            "variant_match_key",
            "fixed_variant_id",
            "fixed_variant_order",
            "fixed_method_label",
        ]
        selected = full.merge(
            FIXED_VARIANT_TABLE[fixed_cols].drop_duplicates("variant_match_key"),
            on="variant_match_key",
            how="inner",
        )

        found_keys = set(selected["variant_match_key"].astype(str))
        expected_keys = FIXED_VARIANT_TABLE["variant_match_key"].astype(str).tolist()
        missing_keys = [key for key in expected_keys if key not in found_keys]
        if missing_keys:
            missing_labels = [FIXED_LABEL_BY_MATCH_KEY.get(key, key) for key in missing_keys]
            message = (
                f"{selection_path}: fixed variants absent from the selection table: "
                f"{missing_labels}"
            )
            if REQUIRE_ALL_FIXED_VARIANTS:
                raise RuntimeError(message)
            print(f"[Warning] {message}")

        selected["method_id"] = selected["variant_match_key"].astype(str)
        selected["method_label"] = selected["fixed_method_label"].astype(str)
        selected["fixed_variant_order"] = pd.to_numeric(
            selected["fixed_variant_order"], errors="coerce"
        )
    else:
        selected = full.copy()
        if "select_for_final" in selected.columns:
            selected = selected[as_bool_series(selected["select_for_final"])].copy()

        if ALWAYS_INCLUDE_RANDOM_SINGLE:
            ref_rows = full[full["strategy"].astype(str).eq(RANDOM_SINGLE_ID)].copy()
            selected = pd.concat([selected, ref_rows], ignore_index=True)

        if ALWAYS_INCLUDE_NAIVE_MEAN_CONTROL:
            naive_rows = full[full["strategy"].astype(str).eq(NAIVE_BASELINE_ID)].copy()
            selected = pd.concat([selected, naive_rows], ignore_index=True)

        if not INCLUDE_S0_AS_METHOD:
            selected = selected[selected["strategy"].astype(str).ne(NAIVE_BASELINE_ID)].copy()

        selected["method_id"] = selected.apply(get_method_id, axis=1)
        selected["method_label"] = selected.apply(get_method_label, axis=1)
        selected["fixed_variant_order"] = selected["strategy_order"]

    if not INCLUDE_S0_AS_METHOD:
        selected = selected[selected["strategy"].astype(str).ne(NAIVE_BASELINE_ID)].copy()

    selected = selected.sort_values(
        ["fixed_variant_order", "strategy_order", "n_metacells", "top_k", "sampled_metacells_k", "method_id"],
        na_position="first",
    ).drop_duplicates(["variant_match_key", "method_id"], keep="first")

    if selected.empty:
        raise RuntimeError(f"No selected variants in {selection_path}")

    # Explicitly verify that the seven requested methods remain distinct.
    expected_n = len(VARIANTS) if USE_FIXED_VARIANTS else None
    observed_n = selected["method_id"].astype(str).nunique()
    if expected_n is not None and observed_n != expected_n:
        diagnostic = selected[[
            c for c in [
                "source_variant_id", "variant_match_key", "method_id",
                "method_label", "fixed_variant_order", "strategy",
                "n_metacells", "sampled_metacells_k", "top_k",
                "n_control_cells_to_average",
            ] if c in selected.columns
        ]].copy()
        raise RuntimeError(
            f"{selection_path}: expected {expected_n} fixed variants but matched "
            f"{observed_n}. Matched rows:\n{diagnostic.to_string(index=False)}"
        )

    return selected.reset_index(drop=True)


def find_seed_table(sub_path: Path, source: str) -> Path | None:
    if source == "control_manifold":
        candidates = CONTROL_MANIFOLD_SEED_TABLE_CANDIDATES
    elif source == "perturbation_effect":
        candidates = PERTURBATION_EFFECT_SEED_TABLE_CANDIDATES
    elif source == "gene_program":
        candidates = GENE_PROGRAM_SEED_TABLE_CANDIDATES
    elif source == "mlp_forward":
        candidates = MLP_FORWARD_SEED_TABLE_CANDIDATES
    elif source == "mlp_inverse":
        candidates = MLP_INVERSE_SEED_TABLE_CANDIDATES
    else:
        raise ValueError(f"Unknown metric source: {source}")
    return first_existing_path([sub_path / p for p in candidates])


def add_metric_value_column(df: pd.DataFrame, metric: str, info: Mapping[str, Any]) -> pd.DataFrame:
    out = df.copy()
    out[metric] = np.nan
    for col in info.get("source_columns", []):
        if col in out.columns:
            vals = pd.to_numeric(out[col], errors="coerce")
            out[metric] = out[metric].where(out[metric].notna(), vals)
    return out


def selected_aggregate_metric_rows(
    selected: pd.DataFrame,
    dataset_id: str,
    group: str,
    metric: str,
    info: Mapping[str, Any],
) -> pd.DataFrame:
    mean_col = next((c for c in info.get("selected_mean_columns", []) if c in selected.columns), None)
    if mean_col is None:
        return pd.DataFrame()

    rows = []
    for _, row in selected.iterrows():
        val = to_float_or_nan(row.get(mean_col))
        if not np.isfinite(val):
            continue
        rows.append({
            "dataset_id": dataset_id,
            "perturbed_group": group,
            "dataset_group_id": make_dataset_group_id(dataset_id, group),
            "dataset_group_label": make_dataset_group_label(dataset_id, group),
            "metric": metric,
            "metric_label": info.get("label", metric),
            "direction": info.get("direction", "lower"),
            "source": info.get("source", "selected_aggregate"),
            "source_path": "selected_variants",
            "method_id": str(row["method_id"]),
            "method_label": str(row["method_label"]),
            "strategy": str(row.get("strategy", "")),
            "strategy_order": to_float_or_nan(row.get("strategy_order")),
            "variant_match_key": str(row.get("variant_match_key", row.get("method_id", ""))),
            "sampling_seed_for_plot": "aggregate",
            "value": val,
            "is_seed_level": False,
        })
    return pd.DataFrame(rows)


def load_metric_rows_for_source(
    sub_path: Path,
    dataset_id: str,
    group: str,
    selected: pd.DataFrame,
    source: str,
    metrics_for_source: Sequence[str],
) -> pd.DataFrame:
    seed_path = find_seed_table(sub_path, source)
    rows: list[pd.DataFrame] = []

    if seed_path is None:
        for metric in metrics_for_source:
            info = METRIC_DEFINITIONS[metric]
            rows.append(selected_aggregate_metric_rows(selected, dataset_id, group, metric, info))
        return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

    raw = read_table(seed_path)
    raw = filter_table_to_context(raw, dataset_id, group)
    if raw.empty:
        return pd.DataFrame()

    raw = canonicalize_variant_table(raw)
    seed_col = infer_seed_column(raw)
    raw = raw.rename(columns={seed_col: "sampling_seed_for_plot"})

    # Map source-table variants to selected method labels.
    selected_map = selected[["variant_match_key", "method_id", "method_label"]].drop_duplicates()
    raw = raw.merge(selected_map, on="variant_match_key", how="inner")

    if raw.empty:
        return pd.DataFrame()

    # Reduce duplicate source rows per selected method and seed.
    meta_cols = [
        "method_id", "method_label", "variant_match_key", "sampling_seed_for_plot",
        "strategy", "strategy_order",
    ]

    for metric in metrics_for_source:
        info = METRIC_DEFINITIONS[metric]
        tmp = add_metric_value_column(raw, metric, info)
        tmp = tmp[tmp[metric].notna()].copy()
        if tmp.empty:
            # Try selected-table aggregate fallback for this metric only.
            rows.append(selected_aggregate_metric_rows(selected, dataset_id, group, metric, info))
            continue

        keep = [c for c in meta_cols if c in tmp.columns] + [metric]
        tmp = tmp[keep].copy()
        tmp = tmp.groupby(
            [c for c in meta_cols if c in tmp.columns],
            dropna=False,
            as_index=False,
        )[metric].mean()

        tmp = tmp.rename(columns={metric: "value"})
        tmp["dataset_id"] = dataset_id
        tmp["perturbed_group"] = group
        tmp["dataset_group_id"] = make_dataset_group_id(dataset_id, group)
        tmp["dataset_group_label"] = make_dataset_group_label(dataset_id, group)
        tmp["metric"] = metric
        tmp["metric_label"] = info.get("label", metric)
        tmp["direction"] = info.get("direction", "lower")
        tmp["source"] = source
        tmp["source_path"] = str(seed_path)
        tmp["is_seed_level"] = True
        rows.append(tmp)

        # If a selected method is absent from the seed-level table, recover its
        # aggregate mean from selected_variants_TEMPLATE_EDIT_ME.csv when available.
        # This is especially important for S0 naive mean control, which may be
        # deterministic and therefore absent from repeated-run seed summaries.
        present_methods = set(tmp["method_id"].astype(str))
        missing_selected = selected[~selected["method_id"].astype(str).isin(present_methods)].copy()
        if not missing_selected.empty:
            fallback = selected_aggregate_metric_rows(
                missing_selected,
                dataset_id=dataset_id,
                group=group,
                metric=metric,
                info=info,
            )
            if fallback is not None and not fallback.empty:
                rows.append(fallback)

    if not rows:
        return pd.DataFrame()
    return pd.concat([r for r in rows if r is not None and not r.empty], ignore_index=True)


def collect_all_metric_rows() -> pd.DataFrame:
    included = [m for m in INCLUDED_METRICS if m in METRIC_DEFINITIONS]
    unknown = sorted(set(INCLUDED_METRICS) - set(included))
    if unknown:
        print(f"[Warning] Ignoring unknown metrics: {unknown}")

    by_source: dict[str, list[str]] = {}
    for metric in included:
        by_source.setdefault(str(METRIC_DEFINITIONS[metric]["source"]), []).append(metric)

    all_rows: list[pd.DataFrame] = []
    sub_paths = discover_sub_paths()
    print(f"[Info] Found {len(sub_paths)} dataset/group paths with selected-variant tables.")

    for dataset_id, group, sub_path in sub_paths:
        selection_path = sub_path / "result_analysis" / SELECTION_TABLE_NAME
        try:
            selected = load_selected_variants(selection_path)
            print(
                f"[Selected] {dataset_id}/{group}: "
                + ", ".join(selected.sort_values("fixed_variant_order")["method_label"].astype(str).tolist())
            )
        except Exception as e:
            print(f"[Skip] {dataset_id}/{group}: failed to load selected variants: {e}")
            continue

        for source, metrics_for_source in by_source.items():
            try:
                rows = load_metric_rows_for_source(sub_path, dataset_id, group, selected, source, metrics_for_source)
            except Exception as e:
                print(f"[Warning] {dataset_id}/{group}/{source}: {e}")
                rows = pd.DataFrame()
            if rows is not None and not rows.empty:
                all_rows.append(rows)

    if not all_rows:
        raise RuntimeError("No metric rows were collected. Check root_dir, dataset_names, groups, and metric table paths.")

    out = pd.concat(all_rows, ignore_index=True)
    out["value"] = pd.to_numeric(out["value"], errors="coerce")
    out = out[out["value"].notna()].copy()

    # If method level is strategy, collapse duplicate variant rows per method/seed/metric.
    if METHOD_LEVEL == "strategy":
        group_cols = [
            "dataset_id", "perturbed_group", "dataset_group_id", "dataset_group_label",
            "metric", "metric_label", "direction",
            "sampling_seed_for_plot", "method_id", "method_label", "strategy", "strategy_order",
        ]
        out = out.groupby(group_cols, dropna=False, as_index=False).agg(
            value=("value", "mean"),
            is_seed_level=("is_seed_level", "max"),
            source=("source", lambda x: ";".join(sorted(set(map(str, x))))),
            source_path=("source_path", "first"),
        )

    return out.reset_index(drop=True)


# ============================================================
# Ranking and critical-distance calculations
# ============================================================

def make_block_id(df: pd.DataFrame, block_cols: Sequence[str]) -> pd.Series:
    return df[list(block_cols)].astype(str).agg("||".join, axis=1)


def average_metric_means(long_df: pd.DataFrame) -> pd.DataFrame:
    """Average seed values first, leaving one value per dataset-group/metric/method."""
    group_cols = [
        "dataset_id", "perturbed_group", "dataset_group_id", "dataset_group_label",
        "metric", "metric_label", "direction",
        "method_id", "method_label", "strategy", "strategy_order",
    ]
    return long_df.groupby(group_cols, dropna=False, as_index=False).agg(
        value=("value", "mean"),
        n_values=("value", "size"),
    )




def keep_best_s5_variant(long_df: pd.DataFrame) -> pd.DataFrame:
    """
    If two or more S5 variants were selected, keep only the better-performing S5
    variant before optional CD exclusion. This makes intermediate tables clean and
    also allows S5 to be included later by removing it from EXCLUDE_STRATEGIES_FROM_CD.

    Better performance is determined by seed-averaged metric values. For lower-is-better
    metrics, smaller values receive better ranks; for higher-is-better metrics, larger
    values receive better ranks.
    """
    if not SELECT_BEST_S5_VARIANT or long_df.empty:
        return long_df

    s5_strategy = "S5_SEACell_OT_sampled_average"
    s5 = long_df[long_df["strategy"].astype(str).eq(s5_strategy)].copy()
    if s5.empty:
        return long_df

    if S5_BEST_SELECTION_SCOPE not in {"dataset_group", "dataset_group_metric"}:
        raise ValueError("S5_BEST_SELECTION_SCOPE must be 'dataset_group' or 'dataset_group_metric'.")

    mean_s5 = s5.groupby(
        ["dataset_group_id", "metric", "direction", "method_id", "method_label"],
        dropna=False,
        as_index=False,
    ).agg(mean_value=("value", "mean"), n_values=("value", "size"))

    if mean_s5.empty:
        return long_df

    ranked_parts: list[pd.DataFrame] = []
    for (dataset_group_id, metric), sub in mean_s5.groupby(["dataset_group_id", "metric"], dropna=False):
        direction = str(sub["direction"].iloc[0])
        ascending = direction == "lower"
        tmp = sub.copy()
        tmp["s5_rank_for_metric"] = tmp["mean_value"].rank(method="average", ascending=ascending)
        ranked_parts.append(tmp)

    ranked = pd.concat(ranked_parts, ignore_index=True) if ranked_parts else pd.DataFrame()
    if ranked.empty:
        return long_df

    keep_rows = []
    if S5_BEST_SELECTION_SCOPE == "dataset_group_metric":
        for (dataset_group_id, metric), sub in ranked.groupby(["dataset_group_id", "metric"], dropna=False):
            best = sub.sort_values(["s5_rank_for_metric", "method_id"], ascending=[True, True]).iloc[0]
            keep_rows.append({
                "dataset_group_id": dataset_group_id,
                "metric": metric,
                "method_id": str(best["method_id"]),
                "method_label": str(best["method_label"]),
            })
    else:
        score = ranked.groupby(
            ["dataset_group_id", "method_id", "method_label"],
            dropna=False,
            as_index=False,
        ).agg(
            mean_s5_rank=("s5_rank_for_metric", "mean"),
            n_metrics=("metric", "nunique"),
            mean_metric_value=("mean_value", "mean"),
        )
        for dataset_group_id, sub in score.groupby("dataset_group_id", dropna=False):
            best = sub.sort_values(
                ["mean_s5_rank", "n_metrics", "method_id"],
                ascending=[True, False, True],
            ).iloc[0]
            keep_rows.append({
                "dataset_group_id": dataset_group_id,
                "metric": None,
                "method_id": str(best["method_id"]),
                "method_label": str(best["method_label"]),
            })

    keep_df = pd.DataFrame(keep_rows)
    if keep_df.empty:
        return long_df

    print("[Info] Best S5 variant retained before CD exclusion:")
    for _, row in keep_df.iterrows():
        metric_part = f" / {row['metric']}" if row.get("metric") is not None else ""
        print(f"  - {row['dataset_group_id']}{metric_part}: {row['method_label']} [{row['method_id']}]")

    non_s5 = long_df[~long_df["strategy"].astype(str).eq(s5_strategy)].copy()

    if S5_BEST_SELECTION_SCOPE == "dataset_group_metric":
        keep_keys = set(zip(
            keep_df["dataset_group_id"].astype(str),
            keep_df["metric"].astype(str),
            keep_df["method_id"].astype(str),
        ))
        s5_keep_mask = [
            (str(dg), str(m), str(mid)) in keep_keys
            for dg, m, mid in zip(s5["dataset_group_id"], s5["metric"], s5["method_id"])
        ]
    else:
        keep_keys = set(zip(
            keep_df["dataset_group_id"].astype(str),
            keep_df["method_id"].astype(str),
        ))
        s5_keep_mask = [
            (str(dg), str(mid)) in keep_keys
            for dg, mid in zip(s5["dataset_group_id"], s5["method_id"])
        ]

    s5_kept = s5.loc[s5_keep_mask].copy()
    return pd.concat([non_s5, s5_kept], ignore_index=True)



def expand_naive_mean_control_across_seed_blocks(long_df: pd.DataFrame) -> pd.DataFrame:
    """
    Replicate deterministic S0 aggregate values across observed seed blocks.

    Many repeated-run tables have seed-level rows for stochastic strategies but
    only one aggregate/deterministic row for S0 naive mean control. CD diagrams
    rank methods within complete blocks, so a single S0 aggregate row would be
    present in a different block from the stochastic seed rows and would often
    be dropped. This function duplicates the S0 value across the observed seeds
    in the same dataset_group × metric so S0 can enter seed-level rankings.
    """
    if not ALWAYS_INCLUDE_NAIVE_MEAN_CONTROL or long_df.empty:
        return long_df

    out_parts: list[pd.DataFrame] = []
    naive_strategy = NAIVE_BASELINE_ID

    for (dataset_group_id, metric), sub in long_df.groupby(["dataset_group_id", "metric"], dropna=False):
        sub = sub.copy()
        naive = sub[sub["strategy"].astype(str).eq(naive_strategy)].copy()
        non_naive = sub[~sub["strategy"].astype(str).eq(naive_strategy)].copy()

        if naive.empty or non_naive.empty:
            out_parts.append(sub)
            continue

        observed_seeds = (
            non_naive["sampling_seed_for_plot"]
            .dropna()
            .astype(str)
            .replace({"nan": np.nan, "None": np.nan})
            .dropna()
            .unique()
            .tolist()
        )
        observed_seeds = [s for s in observed_seeds if s.lower() not in {"aggregate", "nan", "none"}]

        if not observed_seeds:
            out_parts.append(sub)
            continue

        # Use the mean S0 value if multiple S0 rows are available. Preserve metadata
        # from the first S0 row and then duplicate it across all observed seed IDs.
        base = naive.iloc[0].copy()
        base["value"] = float(pd.to_numeric(naive["value"], errors="coerce").mean())
        replicated_rows = []
        for seed in observed_seeds:
            row = base.copy()
            row["sampling_seed_for_plot"] = seed
            row["is_seed_level"] = True
            row["source"] = f"{base.get('source', 'selected_aggregate')};s0_replicated_to_seed_blocks"
            replicated_rows.append(row)

        replicated = pd.DataFrame(replicated_rows)

        # Keep non-S0 rows and the replicated S0 rows. Dropping original aggregate
        # S0 rows avoids extra incomplete seed-level blocks such as "aggregate".
        out_parts.append(pd.concat([non_naive, replicated], ignore_index=True))

    if not out_parts:
        return long_df
    return pd.concat(out_parts, ignore_index=True).reset_index(drop=True)


def apply_cd_method_labels_and_collapses(long_df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply final method IDs and labels used by the CD diagrams.

    This is intentionally applied after keep_best_s5_variant(...), so two selected
    S5 variants are first reduced to the best one and only then collapsed to the
    single S5 method label used in the CD diagram.
    """
    if long_df.empty:
        return long_df

    out = long_df.copy()

    if COLLAPSE_S4_VARIANTS_TO_BASE_METHOD:
        s4 = out["strategy"].astype(str).eq("S4_SEACell_balanced_random_sample")
        out.loc[s4, "method_id"] = "S4_SEACell_balanced_random_sample"
        out.loc[s4, "method_label"] = STRATEGY_PLOT_LABELS["S4_SEACell_balanced_random_sample"]

    if COLLAPSE_S5_AFTER_BEST_SELECTION:
        s5 = out["strategy"].astype(str).eq("S5_SEACell_OT_sampled_average")
        out.loc[s5, "method_id"] = "S5_SEACell_OT_sampled_average"
        out.loc[s5, "method_label"] = STRATEGY_PLOT_LABELS["S5_SEACell_OT_sampled_average"]

    if USE_STRATEGY_PLOT_LABELS_FOR_CD:
        for strategy, label in STRATEGY_PLOT_LABELS.items():
            mask = out["strategy"].astype(str).eq(strategy)
            out.loc[mask, "method_id"] = strategy
            out.loc[mask, "method_label"] = label

    # If multiple rows now map to the same strategy-level method within the same
    # ranking block, downstream groupby operations average them. This is desired
    # for S4, for S5 after best-variant filtering, and for summary CD diagrams
    # where the same strategy may have slightly different selected variant IDs
    # across dataset/groups.
    return out.reset_index(drop=True)


def filter_methods_for_cd(long_df: pd.DataFrame) -> pd.DataFrame:
    """Remove strategies that should not enter CD diagrams, such as S0."""
    if long_df.empty:
        return long_df
    excluded = set(map(str, EXCLUDE_STRATEGIES_FROM_CD))
    if not excluded:
        return long_df
    before = long_df.shape[0]
    out = long_df[~long_df["strategy"].astype(str).isin(excluded)].copy()
    after = out.shape[0]
    removed_methods = sorted(
        long_df.loc[long_df["strategy"].astype(str).isin(excluded), "method_label"]
        .astype(str)
        .unique()
        .tolist()
    )
    print(
        f"[Info] Excluded strategies from CD diagrams: {sorted(excluded)}; "
        f"removed {before - after} rows. Excluded labels: {removed_methods}"
    )
    return out.reset_index(drop=True)

def _select_complete_design(values: pd.DataFrame, title: str) -> tuple[pd.DataFrame, list[str]]:
    """
    scikit-posthocs Conover-Friedman and CD diagrams need a complete block x method matrix.

    When METHOD_LEVEL='variant', selected variants can differ by dataset/group. A summary plot
    can therefore have no complete rows if all variants are forced to appear in every block.
    This helper greedily removes the least-covered methods until enough complete blocks remain.
    """
    work = values.copy().dropna(axis=1, how="all")
    dropped: list[str] = []

    if FORCE_KEEP_ALL_FIXED_VARIANTS and USE_FIXED_VARIANTS:
        expected_method_ids = (
            FIXED_VARIANT_TABLE
            .sort_values("fixed_variant_order")["method_id"]
            .astype(str)
            .drop_duplicates()
            .tolist()
        )
        missing_methods = [m for m in expected_method_ids if m not in work.columns.astype(str)]
        if missing_methods:
            missing_labels = [FIXED_LABEL_BY_MATCH_KEY.get(m, m) for m in missing_methods]
            raise RuntimeError(
                f"{title}: the following requested variants have no values in this CD input: "
                f"{missing_labels}. Refusing to draw a reduced-method figure."
            )

        # Preserve the exact seven-variant order and retain only complete blocks.
        work.columns = work.columns.astype(str)
        work = work.reindex(columns=expected_method_ids)
        complete = work.dropna(axis=0, how="any")
        if complete.shape[0] < MIN_BLOCKS_FOR_CD:
            coverage = work.notna().sum(axis=0).to_dict()
            coverage_named = {FIXED_LABEL_BY_MATCH_KEY.get(k, k): int(v) for k, v in coverage.items()}
            raise RuntimeError(
                f"{title}: only {complete.shape[0]} complete blocks contain all seven requested "
                f"variants; at least {MIN_BLOCKS_FOR_CD} are required. Per-variant coverage: "
                f"{coverage_named}. Refusing to drop a variant."
            )
        return complete, []

    if work.shape[1] < MIN_METHODS_FOR_CD:
        raise RuntimeError(
            f"Insufficient methods before complete-block filtering for {title}: "
            f"n_methods={work.shape[1]}"
        )

    min_coverage = max(1, int(math.ceil(MIN_METHOD_COVERAGE_FRACTION * max(work.shape[0], 1))))
    if min_coverage > 1:
        coverage = work.notna().sum(axis=0)
        low_cov = coverage[coverage < min_coverage].index.astype(str).tolist()
        if low_cov:
            dropped.extend(low_cov)
            work = work.drop(columns=low_cov)

    if COMPLETE_BLOCKS_ONLY:
        while work.shape[1] >= MIN_METHODS_FOR_CD:
            complete = work.dropna(axis=0, how="any")
            if complete.shape[0] >= MIN_BLOCKS_FOR_CD:
                if dropped:
                    print(
                        f"[Info] {title}: dropped {len(dropped)} incompletely covered methods "
                        f"to obtain a complete CD design: {dropped}"
                    )
                return complete, dropped

            coverage = work.notna().sum(axis=0)
            if coverage.empty:
                break
            force_keep = set(map(str, FORCE_KEEP_METHOD_IDS))
            drop_candidates = [c for c in coverage.index.astype(str).tolist() if c not in force_keep]
            if not drop_candidates:
                break
            candidate_cov = coverage.loc[drop_candidates]
            # Remove the least-covered non-forced method; break ties deterministically.
            worst_cov = candidate_cov.min()
            worst_candidates = candidate_cov[candidate_cov == worst_cov].index.astype(str).tolist()
            worst = sorted(worst_candidates)[-1]
            dropped.append(worst)
            work = work.drop(columns=[worst])

        raise RuntimeError(
            f"Insufficient complete ranking blocks for {title}: "
            f"n_blocks={work.dropna(axis=0, how='any').shape[0] if work.shape[1] else 0}, "
            f"n_methods={work.shape[1]}, dropped_methods={dropped}"
        )

    # Non-complete mode still has to return a complete matrix for scikit-posthocs.
    # Here we first keep blocks with at least MIN_METHODS_FOR_CD observations, then
    # apply the same complete-design selection to avoid imputation.
    work = work.dropna(axis=0, thresh=MIN_METHODS_FOR_CD)
    old_complete = COMPLETE_BLOCKS_ONLY
    try:
        globals()["COMPLETE_BLOCKS_ONLY"] = True
        return _select_complete_design(work, title)
    finally:
        globals()["COMPLETE_BLOCKS_ONLY"] = old_complete


def prepare_rank_matrix(
    df: pd.DataFrame,
    block_cols: Sequence[str],
    title: str,
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    """Return average ranks and full rank matrix for a CD plot."""
    if df.empty:
        raise RuntimeError(f"No data for {title}")

    work = df.copy()
    work["block_id"] = make_block_id(work, block_cols)

    # Avoid duplicate method values within a block.
    group_cols = ["block_id", "method_id", "method_label"]
    agg = work.groupby(group_cols, dropna=False, as_index=False).agg(
        value=("value", "mean"),
        direction=("direction", "first"),
    )

    values = agg.pivot_table(index="block_id", columns="method_id", values="value", aggfunc="mean")
    labels = agg.drop_duplicates("method_id").set_index("method_id")["method_label"].to_dict()
    block_direction = agg.drop_duplicates("block_id").set_index("block_id")["direction"].to_dict()

    values, dropped_methods = _select_complete_design(values, title=title)

    ranks = pd.DataFrame(index=values.index, columns=values.columns, dtype=float)
    for block_id, row in values.iterrows():
        direction = block_direction.get(block_id, "lower")
        ascending = direction == "lower"
        ranks.loc[block_id] = row.rank(method="average", ascending=ascending)

    avg = pd.DataFrame({
        "method_id": ranks.columns.astype(str),
        "average_rank": ranks.mean(axis=0).to_numpy(dtype=float),
        "n_blocks": ranks.notna().sum(axis=0).to_numpy(dtype=int),
    })
    avg["method_label"] = avg["method_id"].map(labels).fillna(avg["method_id"])
    avg = avg.sort_values("average_rank", ascending=True).reset_index(drop=True)

    meta = {
        "title": title,
        "n_blocks": int(ranks.shape[0]),
        "n_methods": int(ranks.shape[1]),
        "block_cols": list(block_cols),
        "complete_blocks_only": COMPLETE_BLOCKS_ONLY,
        "dropped_methods_for_complete_design": ";".join(map(str, dropped_methods)),
    }
    return avg, ranks, meta


def conover_critical_distance(k: int, n_blocks: int, alpha: float = 0.05) -> float:
    if k < 2 or n_blocks < 1:
        return np.nan
    try:
        q_alpha = float(stats.studentized_range.ppf(1.0 - alpha, k, np.inf) / np.sqrt(2.0))
    except Exception:
        # Approximate two-sided Conover q_alpha/sqrt(2) values for alpha=0.05.
        # Used only if scipy.stats.studentized_range is unavailable.
        q_table_005 = {
            2: 1.960, 3: 2.344, 4: 2.569, 5: 2.728, 6: 2.850,
            7: 2.949, 8: 3.031, 9: 3.102, 10: 3.164,
            11: 3.219, 12: 3.268, 13: 3.313, 14: 3.354,
            15: 3.391, 16: 3.426, 17: 3.458, 18: 3.489,
            19: 3.517, 20: 3.544,
        }
        if abs(alpha - 0.05) > 1e-12:
            print("[Warning] Fallback q table is for alpha=0.05; using nearest available approximation.")
        q_alpha = q_table_005.get(k, q_table_005[20] + 0.03 * (k - 20))
    return q_alpha * math.sqrt(k * (k + 1) / (6.0 * n_blocks))


def friedman_p_value(rank_matrix: pd.DataFrame) -> float:
    if rank_matrix.shape[1] < 3 or rank_matrix.shape[0] < 2:
        return np.nan
    try:
        arrays = [rank_matrix[c].to_numpy(dtype=float) for c in rank_matrix.columns]
        stat, p = stats.friedmanchisquare(*arrays)
        return float(p)
    except Exception:
        return np.nan


def find_non_significant_groups(avg_ranks: pd.DataFrame, cd: float) -> list[list[str]]:
    """Find maximal contiguous groups whose average-rank range is <= CD."""
    if not np.isfinite(cd):
        return []
    ordered = avg_ranks.sort_values("average_rank", ascending=True).reset_index(drop=True)
    groups_out: list[list[str]] = []
    n = ordered.shape[0]
    for i in range(n):
        best_j = None
        for j in range(n - 1, i, -1):
            if float(ordered.loc[j, "average_rank"] - ordered.loc[i, "average_rank"]) <= cd:
                best_j = j
                break
        if best_j is not None and best_j > i:
            group = ordered.loc[i:best_j, "method_id"].astype(str).tolist()
            # Keep only maximal groups.
            s = set(group)
            if not any(s < set(g) for g in groups_out):
                groups_out = [g for g in groups_out if not set(g) < s]
                groups_out.append(group)
    return groups_out


# ============================================================
# Critical-difference plotting with scikit-posthocs
# ============================================================

def require_scikit_posthocs() -> Any:
    if sp is None:
        raise ImportError(
            "scikit-posthocs is required for this script because plotting uses "
            "scikit_posthocs.critical_difference_diagram. Install it with:\n"
            "    pip install scikit-posthocs\n"
            "or, if using conda/mamba:\n"
            "    conda install -c conda-forge scikit-posthocs"
        )
    return sp


def make_unique_plot_labels(avg_ranks: pd.DataFrame) -> dict[str, str]:
    """Return method_id -> unique display label for scikit-posthocs."""
    labels = avg_ranks.set_index("method_id")["method_label"].astype(str).to_dict()
    counts: dict[str, int] = {}
    used: set[str] = set()
    out: dict[str, str] = {}

    for method_id in avg_ranks["method_id"].astype(str).tolist():
        base = labels.get(method_id, method_id)
        counts[base] = counts.get(base, 0) + 1
        label = base if counts[base] == 1 else f"{base} [{counts[base]}]"
        while label in used:
            counts[base] += 1
            label = f"{base} [{counts[base]}]"
        used.add(label)
        out[method_id] = label
    return out


def compute_conover_pvalue_matrix(rank_matrix: pd.DataFrame, plot_labels: Mapping[str, str]) -> pd.DataFrame:
    """
    Compute Conover-Friedman post-hoc p-values from the block-wise rank matrix.

    rank_matrix rows are benchmark blocks and columns are selected methods.
    Values are already within-block ranks, so the post-hoc test is performed on
    these ranks directly to avoid mixing incompatible metric scales.
    """
    sp_module = require_scikit_posthocs()

    method_ids = [str(c) for c in rank_matrix.columns]
    labels = [str(plot_labels.get(mid, mid)) for mid in method_ids]
    ranked_for_test = rank_matrix.copy()
    ranked_for_test.columns = labels

    try:
        pvals = sp_module.posthoc_conover_friedman(ranked_for_test, p_adjust="holm",)
    except Exception:
        pvals = sp_module.posthoc_conover_friedman(ranked_for_test.to_numpy(dtype=float), p_adjust="holm",)

    pvals = pd.DataFrame(pvals)
    if pvals.shape == (len(labels), len(labels)):
        pvals.index = labels
        pvals.columns = labels
    else:
        raise RuntimeError(
            "Unexpected Conover p-value matrix shape: "
            f"{pvals.shape}; expected {(len(labels), len(labels))}."
        )

    pvals = pvals.apply(pd.to_numeric, errors="coerce")
    for label in labels:
        pvals.loc[label, label] = 1.0
    return pvals


def average_rank_series_for_plot(rank_matrix: pd.DataFrame, plot_labels: Mapping[str, str]) -> pd.Series:
    method_ids = [str(c) for c in rank_matrix.columns]
    labels = [str(plot_labels.get(mid, mid)) for mid in method_ids]
    ranks = rank_matrix.copy()
    ranks.columns = labels
    avg = ranks.mean(axis=0).sort_values(ascending=True)
    avg.name = "average_rank"
    return avg


def plot_critical_difference(
    avg_ranks: pd.DataFrame,
    rank_matrix: pd.DataFrame,
    out_path_base: Path,
    title: str,
    subtitle: str,
) -> dict[str, Any]:
    """Plot a CD diagram using scikit_posthocs.critical_difference_diagram."""
    sp_module = require_scikit_posthocs()

    k = int(avg_ranks.shape[0])
    n_blocks = int(rank_matrix.shape[0])
    cd = conover_critical_distance(k, n_blocks, alpha=ALPHA)
    friedman_p = friedman_p_value(rank_matrix)

    # Keep rank_matrix columns in the same order as avg_ranks.
    ordered_method_ids = avg_ranks.sort_values("average_rank", ascending=True)["method_id"].astype(str).tolist()
    rank_matrix = rank_matrix[[m for m in ordered_method_ids if m in rank_matrix.columns]].copy()

    plot_labels = make_unique_plot_labels(avg_ranks)
    avg_rank_series = average_rank_series_for_plot(rank_matrix, plot_labels)
    sig_matrix = compute_conover_pvalue_matrix(rank_matrix, plot_labels)
    sig_matrix = sig_matrix.loc[avg_rank_series.index, avg_rank_series.index]

    fig_h = CD_FIG_HEIGHT_BASE + CD_FIG_HEIGHT_PER_METHOD * max(k, 4)
    fig_w = max(CD_FIG_WIDTH_BASE, 0.75 * k + 7.0)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=DPI)

    # Do not pass color through label_props. Some scikit-posthocs versions
    # pass color internally to ax.text(), so including color here can raise:
    # "Axes.text() got multiple values for keyword argument 'color'".
    # sp_module.critical_difference_diagram(
    #     ranks=avg_rank_series,
    #     sig_matrix=sig_matrix,
    #     cd=cd,
    #     alpha=ALPHA,
    #     ax=ax,
    #     label_fmt_left="{label} ({rank:.2f})",
    #     label_fmt_right="({rank:.2f}) {label}",
    #     label_props={"fontsize": CD_LABEL_SIZE},
    #     marker_props={"s": CD_MARKER_SIZE},
    #     elbow_props={"linewidth": CD_LINEWIDTH},
    #     crossbar_props={"linewidth": CD_CROSSBAR_LINEWIDTH},
    #     text_h_margin=0.02,
    # )
    # STRATEGY_BASE_COLORS = {
    #     "S0_naive_mean_control_reference": "#839DD1",
    #     "S1_random_single_control": "#6A7FC1",
    #     "S2_random_average_controls": "#6374AE",
    #     "S3_SEACell_metacell_average": "#4A5989",
    #     "S4_SEACell_balanced_random_sample": "#414E6E",
    #     "S5_SEACell_OT_sampled_average": "#7F2667",
    # }
    STRATEGY_BASE_COLORS = {
        "S0_naive_mean_control_reference": "#8B8B8B",
        "S1_random_single_control": "#8ABCD1",
        "S2_random_average_controls": "#66A9C9",
        "S3_SEACell_metacell_average": "#5AA4AE",
        "S4_SEACell_balanced_random_sample": "#3B818C",
        "S5_SEACell_OT_sampled_average": "#7F2667",
    }

    STRATEGY_LABEL_TO_COLOR = {
        STRATEGY_PLOT_LABELS[k]: v
        for k, v in STRATEGY_BASE_COLORS.items()
        if k in STRATEGY_PLOT_LABELS
    }

    METHOD_COLOR_LOOKUP = {
        **STRATEGY_BASE_COLORS,
        **STRATEGY_LABEL_TO_COLOR,
    }

    cd_artists = sp_module.critical_difference_diagram(
        ranks=avg_rank_series,
        sig_matrix=sig_matrix,
        cd=cd,
        alpha=ALPHA,
        ax=ax,
        label_fmt_left="{label} ({rank:.2f})",
        label_fmt_right="({rank:.2f}) {label}",
        label_props={"fontsize": CD_LABEL_SIZE},
        marker_props={"s": CD_MARKER_SIZE},
        elbow_props={"linewidth": CD_LINEWIDTH},
        crossbar_props={"linewidth": CD_CROSSBAR_LINEWIDTH},
        text_h_margin=0.02,
    )

    def _color_for_method_label(text: str) -> str:
        text = str(text)

        for method_label, color in METHOD_COLOR_LOOKUP.items():
            if str(method_label) in text:
                return color

        return CD_COLOR


    # Color text labels by strategy.
    for text_obj in ax.texts:
        label_text = text_obj.get_text()
        text_obj.set_color(_color_for_method_label(label_text))


    # Color marker dots by average-rank position.
    # scikit-posthocs usually stores the dots as PathCollection objects.
    rank_to_color = {
        float(rank): _color_for_method_label(method)
        for method, rank in avg_rank_series.items()
    }

    for collection in ax.collections:
        offsets = collection.get_offsets()
        if offsets is None or len(offsets) == 0:
            continue

        facecolors = []
        edgecolors = []

        for xy in offsets:
            x_pos = float(xy[0])

            nearest_rank = min(
                rank_to_color.keys(),
                key=lambda r: abs(r - x_pos),
            )
            color = rank_to_color.get(nearest_rank, CD_COLOR)

            facecolors.append(color)
            edgecolors.append(color)

        collection.set_facecolors(facecolors)
        collection.set_edgecolors(edgecolors)


    # Keep elbows, axes, and non-significance crossbars neutral.
    for line_obj in ax.lines:
        line_obj.set_color(CD_COLOR)

    # # Apply colors after plotting to avoid keyword conflicts across versions.
    # for text_obj in ax.texts:
    #     text_obj.set_color(CD_COLOR)
    # for line_obj in ax.lines:
    #     line_obj.set_color(CD_COLOR)

    friedman_text = f"; Friedman p={friedman_p:.3g}" if np.isfinite(friedman_p) else ""
    full_subtitle = (
        f"{subtitle}; N={n_blocks} ranking blocks; k={k} methods; "
        f"Conover-Friedman alpha={ALPHA}{friedman_text}"
    )
    ax.set_title(title, fontsize=CD_TITLE_SIZE, weight="bold", pad=24)
    ax.text(
        0.5,
        1.03,
        full_subtitle,
        transform=ax.transAxes,
        ha="center",
        va="bottom",
        fontsize=CD_SUBTITLE_SIZE,
        color="#555555",
    )

    fig.tight_layout()
    out_path_base.parent.mkdir(parents=True, exist_ok=True)

    outputs: dict[str, Any] = {
        "figure_base": out_path_base,
        "n_blocks": n_blocks,
        "n_methods": k,
        "critical_distance": cd,
        "friedman_p": friedman_p,
    }

    # Save the Conover matrix next to the figure. This is the same matrix used
    # by scikit_posthocs.critical_difference_diagram for crossbar grouping.
    if SAVE_INTERMEDIATE_TABLES:
        sig_path = out_path_base.with_name(out_path_base.name + "__conover_pvalues.csv")
        sig_matrix.to_csv(sig_path)
        outputs["conover_pvalues"] = sig_path
        print(f"[Saved] {sig_path}")

    if SAVE_PNG:
        png = out_path_base.with_suffix(".png")
        fig.savefig(png, dpi=DPI, bbox_inches="tight")
        outputs["png"] = png
        print(f"[Saved] {png}")
    if SAVE_SVG:
        svg = out_path_base.with_suffix(".svg")
        fig.savefig(svg, bbox_inches="tight")
        outputs["svg"] = svg
        print(f"[Saved] {svg}")

    if SHOW_FIGURES:
        plt.show()
    else:
        plt.close(fig)

    return outputs


def run_cd_plot(
    df: pd.DataFrame,
    block_cols: Sequence[str],
    out_base: Path,
    title: str,
    subtitle: str,
) -> dict[str, Any] | None:
    try:
        avg, ranks, meta = prepare_rank_matrix(df, block_cols=block_cols, title=title)
    except Exception as e:
        print(f"[Skip CD] {title}: {e}")
        return None

    out_base.parent.mkdir(parents=True, exist_ok=True)
    if SAVE_INTERMEDIATE_TABLES:
        avg.to_csv(out_base.with_name(out_base.name + "__average_ranks.csv"), index=False)
        ranks.to_csv(out_base.with_name(out_base.name + "__rank_matrix.csv"))

    try:
        result = plot_critical_difference(avg, ranks, out_base, title=title, subtitle=subtitle)
    except Exception as e:
        print(f"[Skip CD] {title}: failed to plot with scikit-posthocs: {e}")
        return None

    result.update(meta)
    return result


# ============================================================
# Main workflow
# ============================================================

def main() -> dict[str, Any]:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    long_df_all = collect_all_metric_rows()
    long_df_all = long_df_all[long_df_all["metric"].isin(INCLUDED_METRICS)].copy()

    # Fixed-variant mode keeps all seven requested variants, including both S5
    # settings. The legacy cleanup functions are retained but are no-ops because
    # SELECT_BEST_S5_VARIANT and collapse flags are disabled above.
    long_df_all = keep_best_s5_variant(long_df_all)
    long_df_all = apply_cd_method_labels_and_collapses(long_df_all)
    long_df_all = expand_naive_mean_control_across_seed_blocks(long_df_all)

    # Actual CD diagrams use this filtered table. By default, no strategy is removed;
    # S0 naive mean control is included as a ranked baseline method.
    long_df = filter_methods_for_cd(long_df_all)

    if SAVE_INTERMEDIATE_TABLES:
        all_long_path = OUTPUT_DIR / "fixed_variant_metric_seed_long_table_all_selected.csv"
        long_df_all.to_csv(all_long_path, index=False)
        print(f"[Saved] {all_long_path}")

        long_path = OUTPUT_DIR / "selected_variant_metric_seed_long_table_for_cd.csv"
        long_df.to_csv(long_path, index=False)
        print(f"[Saved] {long_path}")

    mean_df = average_metric_means(long_df)
    if SAVE_INTERMEDIATE_TABLES:
        mean_path = OUTPUT_DIR / "selected_variant_metric_seed_mean_table_for_cd.csv"
        mean_df.to_csv(mean_path, index=False)
        print(f"[Saved] {mean_path}")

    results: list[dict[str, Any]] = []

    # ------------------------------------------------------------------
    # 1. For each dataset/group, plot one CD diagram per metric.
    #    Blocks = random seeds for that metric.
    # ------------------------------------------------------------------
    if MAKE_DATASET_METRIC_CD:
        for dataset_group_id in sorted(long_df["dataset_group_id"].astype(str).unique()):
            dg = long_df[long_df["dataset_group_id"].astype(str).eq(dataset_group_id)].copy()
            if dg.empty:
                continue
            dataset_group_label = str(dg["dataset_group_label"].iloc[0])

            for metric in [m for m in INCLUDED_METRICS if m in set(dg["metric"].astype(str))]:
                sub = dg[dg["metric"].astype(str).eq(metric)].copy()
                if sub.empty:
                    continue
                metric_label = str(sub["metric_label"].iloc[0])
                out_base = (
                    OUTPUT_DIR
                    / "dataset_metric"
                    / safe_filename(dataset_group_id)
                    / f"{safe_filename(dataset_group_id)}__{safe_filename(metric)}__cd"
                )
                res = run_cd_plot(
                    sub,
                    block_cols=["dataset_group_id", "metric", "sampling_seed_for_plot"],
                    out_base=out_base,
                    title=f"{dataset_group_label}: {metric_label}",
                    subtitle="Blocks = random seeds for one metric",
                )
                if res:
                    res["plot_type"] = "dataset_metric_seed_level"
                    res["dataset_group_id"] = dataset_group_id
                    res["dataset_group_label"] = dataset_group_label
                    res["metric"] = metric
                    res["metric_label"] = metric_label
                    results.append(res)

    # ------------------------------------------------------------------
    # 2. For each dataset/group, plot one summary CD diagram.
    #    Blocks = metrics after averaging random seeds.
    # ------------------------------------------------------------------
    if MAKE_DATASET_SUMMARY_CD:
        for dataset_group_id in sorted(mean_df["dataset_group_id"].astype(str).unique()):
            sub = mean_df[mean_df["dataset_group_id"].astype(str).eq(dataset_group_id)].copy()
            if sub.empty:
                continue
            dataset_group_label = str(sub["dataset_group_label"].iloc[0])
            out_base = OUTPUT_DIR / "dataset_summary" / f"{safe_filename(dataset_group_id)}__summary_cd"
            res = run_cd_plot(
                sub,
                block_cols=["dataset_group_id", "metric"],
                out_base=out_base,
                title=f"{dataset_group_label}: summary critical difference",
                subtitle="Blocks = metrics after averaging random seeds",
            )
            if res:
                res["plot_type"] = "dataset_summary_metric_mean"
                res["dataset_group_id"] = dataset_group_id
                res["dataset_group_label"] = dataset_group_label
                res["metric"] = "ALL_INCLUDED_METRICS"
                res["metric_label"] = "ALL_INCLUDED_METRICS"
                results.append(res)

    # ------------------------------------------------------------------
    # 3. One summary CD diagram across all dataset/groups.
    #    Blocks = dataset/group × metric after averaging random seeds.
    # ------------------------------------------------------------------
    if MAKE_ALL_DATASETS_SUMMARY_CD:
        out_base = OUTPUT_DIR / "summary" / "all_dataset_groups__summary_cd"
        res = run_cd_plot(
            mean_df,
            block_cols=["dataset_group_id", "metric"],
            out_base=out_base,
            title="Summary critical difference across all dataset-groups",
            subtitle="Blocks = dataset-group × metric after averaging random seeds",
        )
        if res:
            res["plot_type"] = "all_dataset_groups_summary_metric_mean"
            res["dataset_group_id"] = "ALL"
            res["dataset_group_label"] = "ALL"
            res["metric"] = "ALL_INCLUDED_METRICS"
            res["metric_label"] = "ALL_INCLUDED_METRICS"
            results.append(res)

    # Optional summary CD directly using seed-level blocks.
    if MAKE_ALL_DATASETS_SEED_LEVEL_CD:
        out_base = OUTPUT_DIR / "summary" / "all_dataset_groups__seed_level_cd"
        res = run_cd_plot(
            long_df,
            block_cols=["dataset_group_id", "metric", "sampling_seed_for_plot"],
            out_base=out_base,
            title="Summary seed-level critical difference across all dataset-groups",
            subtitle="Blocks = dataset-group × metric × random seed",
        )
        if res:
            res["plot_type"] = "all_dataset_groups_seed_level"
            res["dataset_group_id"] = "ALL"
            res["dataset_group_label"] = "ALL"
            res["metric"] = "ALL_INCLUDED_METRICS"
            res["metric_label"] = "ALL_INCLUDED_METRICS"
            results.append(res)

    summary_df = pd.DataFrame(results)
    if not summary_df.empty:
        summary_path = OUTPUT_DIR / "critical_difference_plot_summary.csv"
        summary_df.to_csv(summary_path, index=False)
        print(f"[Saved] {summary_path}")
    else:
        print("[Warning] No CD plots were generated. Check the saved long/mean tables and method coverage.")

    return {
        "long_df_all_selected_after_s5_best": long_df_all,
        "long_df_for_cd": long_df,
        "mean_df_for_cd": mean_df,
        "plot_summary": summary_df,
        "output_dir": OUTPUT_DIR,
    }


if __name__ == "__main__":
    main()
